# 09 Single-Board Virtual Instrument

Stage 5a entry point for the Stage 5 overlay: ADC0 time preview, debug FFT spectrum, DAC tone control, FIFO/TX dry-run status, and SPEC header capture.

In [ ]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output

candidate_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path('/home/xilinx/jupyter_notebooks/t510_fengine'),
    Path('/home/xilinx/t510_fengine_bringup'),
    Path('/home/astrolab/demo-ant'),
]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'overlay' / 't510_fengine.bit').exists() and (root / 'python' / 't510_fengine.py').exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find overlay/t510_fengine.bit and python/t510_fengine.py')

sys.path.insert(0, str(PROJECT_ROOT))
from python.packet import STREAM_SPEC, FLAG_UDP_DRY_RUN, FLAG_INTERNAL_EPOCH
from python.t510_fengine import T510FEngine

BITFILE = PROJECT_ROOT / 'overlay' / 't510_fengine.bit'
EXPECTED_CORE_VERSION = 0x00010003
print('Python:', sys.executable)
print('Project root:', PROJECT_ROOT)
print('Bitfile:', BITFILE)
print(f'Expected core version: 0x{EXPECTED_CORE_VERSION:08x}')

In [ ]:
state = {'core': None, 'last_preview': None, 'last_spectrum': None, 'last_header': None}

mask_widget = W.Dropdown(
    options=[('ADC0 only: 0x0001', 0x0001), ('ADC0 I/Q candidate: 0x0003', 0x0003), ('All ports: 0xffff', 0xffff)],
    value=0x0001,
    description='ADC mask',
    style={'description_width': '120px'},
)
tone_enable_widget = W.Checkbox(value=True, description='DAC tone enable')
tone_amplitude_widget = W.IntSlider(value=2048, min=0, max=8192, step=128, description='Amplitude', style={'description_width': '120px'})
tone_phase_step_widget = W.Text(value='0x00800000', description='Phase step', style={'description_width': '120px'})
timeout_widget = W.FloatSlider(value=2.0, min=0.2, max=10.0, step=0.2, description='Timeout s', style={'description_width': '120px'})
download_widget = W.Checkbox(value=True, description='Download bitstream on load')

load_button = W.Button(description='Load overlay', button_style='primary')
init_button = W.Button(description='Init Stage 5', button_style='success')
status_button = W.Button(description='Read status')
tone_button = W.Button(description='Apply DAC tone')
preview_button = W.Button(description='ADC0 scope')
spectrum_button = W.Button(description='Debug FFT')
header_button = W.Button(description='Capture TX header')
stop_button = W.Button(description='Stop')
out = W.Output(layout={'border': '1px solid #ccc', 'padding': '8px'})


def parse_int_text(text):
    return int(str(text).replace('_', ''), 0)


def core():
    if state['core'] is None:
        raise RuntimeError('Click Load overlay first')
    return state['core']


def print_status(status):
    keys = [
        'core_version', 'sync_config', 'status', 'streaming', 'fsm_state',
        'rfdc_status_flags', 'rfdc_current_valid_mask', 'rfdc_seen_valid_mask', 'rfdc_sample_count',
        'spec_packet_count', 'spec_udp_byte_count', 'spec_seq_no', 'spec_frame_id', 'spec_chan0',
        'tx_link_status_flags', 'qsfp_link_up', 'udp_dry_run', 'tx_dry_run_packet_count', 'tx_dry_run_byte_count',
        'tx_fifo_level_words', 'tx_fifo_high_water_words', 'tx_fifo_backpressure_cycles',
        'tx_header_capture_status', 'tx_header_capture_valid', 'tx_header_capture_word_count',
        'debug_status', 'debug_done', 'debug_error', 'debug_peak_bin', 'debug_peak_power',
        'preview_status', 'preview_done', 'preview_sample0', 'dac_enable_mask', 'dac_tone_amplitude', 'dac_tone_phase_step',
    ]
    hex_keys = {'core_version', 'sync_config', 'status', 'rfdc_status_flags', 'tx_link_status_flags', 'tx_header_capture_status', 'debug_status', 'preview_status', 'dac_enable_mask', 'dac_tone_phase_step'}
    for key in keys:
        if key not in status:
            continue
        value = int(status[key])
        if key in hex_keys or 'mask' in key:
            print(f'{key:32s}: 0x{value:08x}')
        else:
            print(f'{key:32s}: {value}')
    if int(status.get('core_version', 0)) != EXPECTED_CORE_VERSION:
        print(f'WARNING: expected core_version 0x{EXPECTED_CORE_VERSION:08x}')


def on_load(_):
    with out:
        clear_output(wait=True)
        print('Loading overlay...')
        state['core'] = T510FEngine(str(BITFILE), download=bool(download_widget.value))
        print('Overlay loaded.')
        print_status(core().read_status())


def on_init(_):
    with out:
        clear_output(wait=True)
        print('Configuring tcxo_10mhz + free_run + mode=spec')
        status = core().init_lab_rfdc(
            mask=int(mask_widget.value),
            mode='spec',
            tone_enable=bool(tone_enable_widget.value),
            tone_amplitude=int(tone_amplitude_widget.value),
            tone_phase_step=parse_int_text(tone_phase_step_widget.value),
            wait_seconds=float(timeout_widget.value),
        )
        print_status(status)


def on_status(_):
    with out:
        clear_output(wait=True)
        print_status(core().read_status())


def on_tone(_):
    with out:
        clear_output(wait=True)
        step = parse_int_text(tone_phase_step_widget.value)
        core().set_dac_tone(enable=bool(tone_enable_widget.value), amplitude=int(tone_amplitude_widget.value), phase_step=step)
        print(f'DAC tone applied: enable={tone_enable_widget.value}, amplitude={tone_amplitude_widget.value}, phase_step=0x{step:08x}')
        print_status(core().read_status())


def on_preview(_):
    with out:
        clear_output(wait=True)
        preview = core().capture_preview(n=1024, input_mask=0x01, timeout=float(timeout_widget.value))
        state['last_preview'] = preview
        samples = np.asarray(preview['iq'][0])
        print(f'ADC0 preview: count={len(samples)}, sample0={preview["sample0"]}')
        print(f'I min/max/std: {samples[:,0].min()} / {samples[:,0].max()} / {samples[:,0].std():.2f}')
        print(f'Q min/max/std: {samples[:,1].min()} / {samples[:,1].max()} / {samples[:,1].std():.2f}')
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(samples[:, 0], label='I')
        ax.plot(samples[:, 1], label='Q')
        ax.set_title('ADC0 time preview')
        ax.set_xlabel('sample index')
        ax.set_ylabel('ADC code')
        ax.grid(True)
        ax.legend()
        plt.show()


def on_spectrum(_):
    with out:
        clear_output(wait=True)
        spec = core().capture_spectrum(timeout=float(timeout_widget.value))
        state['last_spectrum'] = spec
        power = np.asarray(spec['power'])
        freq_hz = np.asarray(spec['freq_hz'])
        print(f'Debug FFT: peak_bin={spec["peak_bin"]}, peak_power={spec["peak_power"]}')
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(freq_hz, power)
        ax.set_title('1024-point debug FFT power')
        ax.set_xlabel('frequency (Hz), unshifted')
        ax.set_ylabel('power')
        ax.grid(True)
        plt.show()


def on_header(_):
    with out:
        clear_output(wait=True)
        capture = core().capture_tx_header(timeout=float(timeout_widget.value))
        state['last_header'] = capture
        header = capture['header']
        print(json.dumps(capture['header_dict'], indent=2, sort_keys=True))
        print('axis_words_first10:')
        for idx, word in enumerate(capture['axis_words'][:10]):
            print(f'  {idx:02d}: 0x{word:016x}')
        checks = [
            ('version == 2', header.version == 2),
            ('stream_type == SPEC', header.stream_type == STREAM_SPEC),
            ('UDP_DRY_RUN flag', bool(header.flags & FLAG_UDP_DRY_RUN)),
            ('INTERNAL_EPOCH flag', bool(header.flags & FLAG_INTERNAL_EPOCH)),
            ('payload_bytes == 8192', header.payload_bytes == 8192),
        ]
        print('checks:')
        for label, ok in checks:
            print(f'  {label}: {"PASS" if ok else "FAIL"}')


def on_stop(_):
    with out:
        clear_output(wait=True)
        core().stop()
        time.sleep(0.05)
        print_status(core().read_status())


load_button.on_click(on_load)
init_button.on_click(on_init)
status_button.on_click(on_status)
tone_button.on_click(on_tone)
preview_button.on_click(on_preview)
spectrum_button.on_click(on_spectrum)
header_button.on_click(on_header)
stop_button.on_click(on_stop)

controls = W.VBox([
    W.HBox([load_button, init_button, status_button, stop_button]),
    W.HBox([mask_widget, timeout_widget, download_widget]),
    W.HBox([tone_enable_widget, tone_amplitude_widget, tone_phase_step_widget, tone_button]),
    W.HBox([preview_button, spectrum_button, header_button]),
    out,
])
display(controls)